###### *DD-58*: Export DataTape Implementation on Cloud
###### Repos affected : gr.cepal.dna.de.delivery

##### Steps
*  Merge the current pull request (dev/qa/prod)
*  Change ** run as ** in 'DataTape' Job in Prod
*  Create Volume ** p-cdh-delivery.internalreports.datatape **
*  Transfer tables to Production
     *  reference.thesis_land_registry_mapping
     *  reference.thesis_county_court_mapping

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS `p-cdh-delivery`.internalreports.datatape

In [0]:
%sql

create or replace table `p-cdh`.reference.thesis_land_registry_mapping as
select * from `d-cdh`.reference.thesis_land_registry_mapping

In [0]:
%sql
create or replace table `p-cdh`.reference.thesis_county_court_mapping as
select * from `d-cdh`.reference.thesis_county_court_mapping

In [0]:
%sql
create or replace table `p-cdh`.reference.datetape_hie as
select * from `d-cdh`.reference.datetape_hie

In [0]:
%sql

INSERT INTO `p-cdh`.workflow.externalfilesconfiguration (
    RelativePath,
    UnityCatalog,
    UnitySchema,
    TableName,
    Flow,
    ValidatorScript,
    NotificationEmails,
    string_columns
)
VALUES ('DataTape', 'p-reportinputfiles', 'internal', 'PoolPerimeter_Snapshot', 'Snapshot', null, null, 'AgreementKey');

In [0]:
%sql

INSERT INTO `p-cdh`.workflow.externalfilesconfiguration (
    RelativePath,
    UnityCatalog,
    UnitySchema,
    TableName,
    Flow,
    ValidatorScript,
    NotificationEmails,
    string_columns
)
VALUES ('DataTape', 'p-reportinputfiles', 'internal', 'PoolPerimeter_Incremental', 'Incremental', null, null, 'AgreementKey');

#### DD-163: Generalli Implementation on Cloud
Repos affected : gr.cepal.dna.de.delivery
#### Steps
* Merge the current pull request (dev/qa/prod)
* Transfer reference.GeneralliPerimeter table to Production

In [0]:
%sql
create or replace table `p-cdh`.reference.GeneralliPerimeter as
select * from `d-cdh`.reference.GeneralliPerimeter

#### DD-163: Generalli Implementation on Cloud
Repos affected : gr.cepal.dna.de.lng-lnd-cup
#### Steps
Merge the current pull request (dev/qa/prod)
* Create Volume in production ( /Volumes/p-cdh/landed/customerspecialcases/ )
* Transfer reference.grouped_special_cases_category_mapping table to Production
* Insert in externalfilesconfiguration

In [0]:
%sql
create or replace table `p-cdh`.reference.grouped_special_cases_category_mapping as
select * from `d-cdh`.reference.grouped_special_cases_category_mapping

In [0]:
%sql
INSERT INTO `p-cdh`.workflow.externalfilesconfiguration (
    RelativePath,
    UnityCatalog,
    UnitySchema,
    TableName,
    Flow,
    ValidatorScript,
    NotificationEmails,
    string_columns
)
VALUES ('CustomerSpecialCases', 'p-reportinputfiles', 'internal', null, 'Incremental', null, null, 'BorrowerNep,SpecialCases');

In [0]:
%sql
INSERT INTO `p-cdh`.workflow.externalfilesconfiguration (
    RelativePath,
    UnityCatalog,
    UnitySchema,
    TableName,
    Flow,
    ValidatorScript,
    NotificationEmails,
    string_columns
)
VALUES ('CustomerSpecialCases', 'p-reportinputfiles', 'internal', null, 'Snapshot', null, null, 'BorrowerNep,SpecialCases');

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS `p-cdh`.landed.customerspecialcases

#### REPBTC-66: Update Portfolioname & CDI for qca and Portfolioname for cti_ivr
Repos affected : gr.cepal.dna.de.lng-lnd-cup
#### Steps
* Merge the current pull request (dev/qa/prod)
* Re run primaryactive.dialer for all businessdates

#### DD-188 - Final Commit

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.reference.aml_report AS SELECT * FROM `qa-cdh`.reference.aml_report;

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.reference.execution_calendar AS SELECT * FROM `qa-cdh`.reference.execution_calendar;

#### DT-48

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.reference.lookup_export AS
SELECT * FROM `qa-cdh`.reference.lookup_export;
 

In [0]:
%sql
UPDATE `p-cdh`.reference.lookup_export
SET Catalog = 'p-cdh-delivery';

#### DT-41

In [0]:
%sql
MERGE INTO
  `p-cdh`.derivativeactive.dexposure AS A
USING (
  SELECT DISTINCT
    ProductKey,
    BusinessDate
  FROM
    `p-cdh`.primary.Product
  WHERE
    SourceSystem = 'profits'
    AND ProductDescription ILIKE '%ΥΠΟ ΑΙΡΕΣΗ%'
) AS B
ON
  A.ProductKey = B.ProductKey
  and A.BusinessDate = B.BusinessDate 
WHEN MATCHED THEN UPDATE SET
  A.conditionalwriteoffamount = A.OutstandingTotalBalance, A.conditionalwriteoffdate = A.opendate

###Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivativearchive','primaryarchive')
      AND table_type='VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)

        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f"Successfully refreshed view: {full_view_name}")

    except Exception as e:
        print(f"   Failed to refresh view: {full_view_name}")


In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivativeactive','primaryactive')
      AND table_type='VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)

        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f"Successfully refreshed view: {full_view_name}")

    except Exception as e:
        print(f"   Failed to refresh view: {full_view_name}")


###Closed loans function update

In [0]:
env='p'
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `p-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0
AND coalesce(B.JustificationCode,0) != 43023

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')

In [0]:
import logging
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import trim

# Configure logging
logging.basicConfig(level=logging.INFO)

spark = SparkSession.builder.appName("example").getOrCreate()

def load_dataframe_from_sql_server(
    server: str,
    table: str
) -> DataFrame:
    # Define the server IP based on the provided server type
    if server == 'dwhcore':
        server_ip = 'jdbc:sqlserver://10.240.2.230:1433'
        user_name = 'DEUser'
        password = 'MM998fqLvS9H2'
    elif server == 'dwhdatamarts':
        server_ip = 'jdbc:sqlserver://10.240.2.231:1433'
        user_name = 'PBIAgent'
        password = 'PB1Ag3nt!'
    elif server == 'CHPROFDBNEW':
        server_ip = 'jdbc:sqlserver://10.240.3.90:1433'
        user_name = 'dna_aws'
        password = 'P@1nl3$$'
    elif server == 'AKTPRODDB16':
        server_ip = 'jdbc:sqlserver://10.240.103.16:1433'
        user_name = 'dna_aws'
        password = 'P@1nl3$$'
    elif server == 'dwhcoreuat':
        server_ip = 'jdbc:sqlserver://10.240.2.227:1533'
        user_name = 'DEUser'
        password = 'MM998fqLvS9H2'
    else:
        raise ValueError("Only available servers are: dwhcore, dwhdatamarts, CHPROFDBNEW, AKTPRODDB16, dwhcoreuat") 
    # Read data from SQL Server using Spark JDBC
    try:
        df = (
            spark.read.format("jdbc") 
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .option("url", f"{server_ip};encrypt=false")
            .option("user", user_name)
            .option("password", password)
            .option("dbtable", table)
            .load()
        )
        logging.info(f"Successfully loaded data from {server} server.")
        return df
    except Exception as e:
        logging.error(f"Error loading data from {server} server: {str(e)}")
        raise



In [0]:
Product = load_dataframe_from_sql_server("dwhcore","dwh.ast.Product")
Loan = load_dataframe_from_sql_server("dwhcore","dwh.ast.Loan")
SGKAsset = load_dataframe_from_sql_server("dwhcore","dwh.adm.SGKAsset")
Classification = load_dataframe_from_sql_server("dwhcore","dwh.cls.Classification")
ClassificationScheme = load_dataframe_from_sql_server("dwhcore","dwh.cls.ClassificationScheme")

ClassificationScheme.createOrReplaceTempView("ClassificationScheme")
Classification.createOrReplaceTempView("Classification")
SGKAsset.createOrReplaceTempView("SGKAsset")
Loan.createOrReplaceTempView("Loan")
Product.createOrReplaceTempView("Product")


In [0]:
dates_df = spark.sql("""
    SELECT DISTINCT CAST(BusinessDate AS STRING) as BusinessDate
    FROM `p-cdh`.derivativearchive.DExposure
    order by businessdate asc
""")
dates = [row['BusinessDate'] for row in dates_df.collect()]


In [0]:
for run_date in dates:
    fix = spark.sql(f'''
                        select distinct p.subproductdesc , sgk.assetkey1,cl.Description as source_system  from Product p
                        left join Loan l on p.productid=l.productid  and l.effectivedate <=  {run_date!r} and l.enddate>  {run_date!r} 
                        left join SGKAsset sgk on sgk.Assetid=l.loanid 
                        left join Classification cl
                        ON cl.ClassificationId = sgk.SourceSystemId
                        where  p.effectivedate <=  {run_date!r} and p.enddate >  {run_date!r} 

        ''')

    fix.createOrReplaceTempView("fix")
        
    

    spark.sql(f"""
        MERGE INTO `p-cdh`.derivativearchive.dexposure      AS t
        USING      fix                              AS s
        ON  t.ExposureKey  = s.assetkey1
        AND t.BusinessDate = {run_date!r}               
        WHEN MATCHED THEN
            UPDATE SET t.SubProductDescription = s.subproductdesc
        """)

    print("SubProductDescription updated.")